# AutoCode Analyzer -- Experiment Notebook

Walks through the pipeline one stage at a time -- exactly what `src/codeanalyzer/pipeline.py` does end to end -- so you can see what each stage produces.

Before running: create `.env` in the project folder with your free Groq key (`GROQ_API_KEY=gsk_...`, from https://console.groq.com/keys) and run `python check_setup.py` once. The embedding cell downloads a ~90MB model the first time only.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make `src/` importable whether you opened Jupyter in the project folder or in notebook/
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from codeanalyzer.config import Config
from codeanalyzer.data_ingestion.repo_loader import clone_repo, load_repo_files, parse_github_url
from codeanalyzer.text_splitter.code_splitter import split_documents
from codeanalyzer.embeddings.embedding_manager import get_embedding_model
from codeanalyzer.vectorstore.chroma_store import CodeVectorStore
from codeanalyzer.llm.groq_llm import get_llm
from codeanalyzer.qa.qa_chain import CodeQAChain

config = Config()
print('Groq key:', 'OK' if config.groq_key_problem() is None else config.groq_key_problem())
print('Repos folder:', config.repos_dir)
print('Vector store folder:', config.chroma_persist_dir)

## 1. Clone a repository

Any form of GitHub link works (repo URL, `.git`, `/tree/...` link).

In [ ]:
REPO_URL = 'https://github.com/kennethreitz/samplemod'  # small repo, good for a quick test

repo = parse_github_url(REPO_URL)
repo_path = clone_repo(repo.web_url, base_path=config.repos_dir)
print(repo.owner, '/', repo.name, '->', repo_path)

## 2. Load source files as LangChain `Document`s

Skipped automatically: `.git`, `node_modules`, virtualenvs, build output, lockfiles, minified bundles, binaries and oversized files.

In [ ]:
documents = load_repo_files(
    repo_path,
    allowed_extensions=config.allowed_extensions,
    ignored_dirs=config.ignored_dirs,
    max_file_size_kb=config.max_file_size_kb,
    max_files=config.max_files,
)
print(f'Loaded {len(documents)} files')
for doc in documents:
    print(' -', doc.metadata['source'], f"({len(doc.page_content)} chars)")

## 3. Context-aware splitting

Each file is split with a splitter tuned for *its own* language, so chunks respect function/class boundaries. Chunks are numbered per file.

In [ ]:
chunks = split_documents(documents, chunk_size=config.chunk_size, chunk_overlap=config.chunk_overlap)
print(f'{len(documents)} files -> {len(chunks)} chunks')
sample = chunks[0]
print('\n--- sample chunk ---')
print(sample.metadata)
print(sample.page_content[:400])

## 4. Embed the chunks (local, free, no API key)

In [ ]:
embeddings = get_embedding_model(config.embedding_model)
vector = embeddings.embed_query(sample.page_content)
print('embedding model:', config.embedding_model)
print('dimensions:', len(vector))
print('first values:', [round(v, 4) for v in vector[:5]])

## 5. Build the Chroma vector store

Re-running this cell rebuilds the collection cleanly (no duplicate chunks).

In [ ]:
store = CodeVectorStore(
    persist_directory=config.chroma_persist_dir,
    collection_name='notebook-experiment',
    embedding_function=embeddings,
).build_from_documents(chunks)
print('chunks stored:', store.count())

retriever = store.as_retriever(k=config.top_k)
for doc in retriever.invoke('helper function'):
    print(doc.metadata['source'], '| part', doc.metadata['chunk_index'] + 1, 'of', doc.metadata['total_chunks'])

## 6. Ask a question (Groq LLM)

In [ ]:
llm = get_llm(config)
qa = CodeQAChain(retriever=retriever, llm=llm, max_history_turns=config.max_history_turns)

result = qa.ask('What does this repository do, and what does the helpers module contain?')
print(result['answer'])
print('\nSOURCES:', result['sources'])

## 7. Follow-up question (multi-turn)

The follow-up is first rewritten into a standalone question using the chat history, so words like "that" resolve correctly before retrieval.

In [ ]:
history = [('human', 'What does this repository do, and what does the helpers module contain?'),
           ('ai', result['answer'])]

follow_up = qa.ask('Where exactly is that defined?', chat_history=history)
print('Standalone question:', follow_up['standalone_question'])
print()
print(follow_up['answer'])
print('\nSOURCES:', follow_up['sources'])

## Next steps

`app.py` exposes this same pipeline through `/ingest` and `/chat`; see the README for running it locally, in Docker, and deployed to AWS via the CI/CD pipeline.